# Train2: fine-tune FunctionGemma on Kaggle

1. Upload `training/Train2/colab.zip` (made by `bash training/pack_colab.sh`) as a Kaggle Dataset named `train2`, then add it to this notebook (Add Input).
2. Settings: Accelerator **GPU** (T4 x2 or P100), Internet **On**, Persistence **Files only**.
3. Save Version > **Save & Run All (Commit)**. It keeps running if you close the tab (12 h limit).

When it finishes, download `train2_result.zip` from the version's Output tab.


In [ ]:
%%bash
set -euo pipefail
ls -R /kaggle/input | head -30   # shows where the dataset is mounted
rm -rf /kaggle/working/fc && mkdir -p /kaggle/working/fc
# the mount path varies (/kaggle/input/<slug> or /kaggle/input/datasets/<user>/<slug>), so search for it
zip=$(find /kaggle/input -name colab.zip | head -1)
if [ -n "$zip" ]; then
  unzip -q "$zip" -d /kaggle/working/fc
else
  train=$(find /kaggle/input -path '*/training/train.py' | head -1)
  [ -n "$train" ] || { echo "error: dataset not attached, use Add Input and pick train2" >&2; exit 1; }
  cp -r "${train%/training/train.py}"/. /kaggle/working/fc/
fi
test -f /kaggle/working/fc/training/train.py


In [ ]:
# keep Kaggle's torch
!pip install -q transformers==5.22.2 trl==1.17.1 datasets==5.1.0 accelerate==1.15.0

In [ ]:
!nvidia-smi

In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/fc
# output lives in /kaggle/working, which a committed run keeps
mkdir -p /kaggle/working/train2_out
ln -sfn /kaggle/working/train2_out training/Train2/output
# under 20 GB (a T4 has 15 GB and no bf16): recompute activations to fit
mem=$(nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits | head -1)
if [ "$mem" -lt 20480 ]; then flag=--gradient-checkpointing; else flag=--no-gradient-checkpointing; fi
resume=
if ls -d training/Train2/output/functiongemma/checkpoint-* >/dev/null 2>&1; then
  resume=--resume
  echo "checkpoint found: resuming"
fi
# T4 x2 shows two GPUs and SFTTrainer rejects DataParallel, so use one
export CUDA_VISIBLE_DEVICES=0
python training/train.py $flag $resume 2>&1 | tee -a training/Train2/output/train.log
test -f training/Train2/output/functiongemma/model.safetensors

In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/fc
test -f training/Train2/output/functiongemma/model.safetensors
export CUDA_VISIBLE_DEVICES=0
python training/evaluate.py --models base,train2 --device cuda 2>&1 | tee training/Train2/output/eval.log

In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/fc
test -f training/Train2/output/functiongemma/model.safetensors
rm -f /kaggle/working/train2_result.zip
zip -r -q /kaggle/working/train2_result.zip training/Train2/output/functiongemma training/Train2/output/eval_*.jsonl training/Train2/output/train.log training/Train2/output/eval.log -x 'training/Train2/output/functiongemma/checkpoint-*'
ls -lh /kaggle/working/train2_result.zip